# 📈 Simple Linear Regression: Salary Data Analysis

> **Objective:** Predict employee **Salary** (USD) from **Years of Experience** using Simple Linear Regression.

| Detail | Value |
|---|---|
| Dataset | `Salary_Data.csv` |
| Independent Variable (X) | `YearsExperience` |
| Dependent Variable (Y) | `Salary` |
| Algorithm | Simple Linear Regression (OLS) |
| Libraries | pandas, numpy, matplotlib, seaborn, scikit-learn |

---

## 1. 📥 Data Loading & Exploration

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# ── Global plot style ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
PALETTE = {'train': '#4C72B0', 'test': '#DD8452', 'line': '#2ca02c'}

# ── Output directory (works locally and on Colab) ─────────────────────────────
OUT_DIR = '/content/' if os.path.exists('/content') else './output/'
os.makedirs(OUT_DIR, exist_ok=True)

print('✅ All libraries imported successfully.')
print(f'📁 Output directory: {OUT_DIR}')

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
# Adjust path as needed; on Colab upload the file first.
CSV_PATH = 'Salary_Data.csv'

df = pd.read_csv(CSV_PATH)

# ── Auto-detect X and Y columns ───────────────────────────────────────────────
# Convention: first column → X (feature), last column → Y (target)
X_COL = df.columns[0]   # 'YearsExperience'
Y_COL = df.columns[-1]  # 'Salary'

print(f'📌 Independent variable (X) : {X_COL}')
print(f'📌 Dependent   variable (Y) : {Y_COL}')
print(f'📐 Dataset shape            : {df.shape}')
print(f'🔢 Data types:\n{df.dtypes}\n')

df.head()

In [ ]:
df.tail()

In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────────
df.describe().T.style.background_gradient(cmap='Blues').format('{:.2f}')

In [ ]:
# ── Missing value audit ───────────────────────────────────────────────────────
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)

# Drop rows with any missing numeric value (robust for production)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'\n✅ Clean dataset shape: {df.shape}')

## 2. 🔍 Data Visualization (Baseline)

Before modelling we visually confirm:
1. **Linearity** – scatter of X vs Y should show a roughly straight-line trend.
2. **Distribution** – histograms reveal skewness / outliers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 1. Scatter: X vs Y ────────────────────────────────────────────────────────
ax = axes[0]
ax.scatter(df[X_COL], df[Y_COL], color=PALETTE['train'],
           edgecolors='white', s=90, linewidths=0.7, alpha=0.85)
ax.set_title(f'{X_COL} vs {Y_COL}', fontweight='bold')
ax.set_xlabel(X_COL)
ax.set_ylabel(Y_COL)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── 2. Histogram of X ────────────────────────────────────────────────────────
ax = axes[1]
ax.hist(df[X_COL], bins=10, color=PALETTE['train'],
        edgecolor='white', alpha=0.85)
ax.set_title(f'Distribution of {X_COL}', fontweight='bold')
ax.set_xlabel(X_COL)
ax.set_ylabel('Frequency')

# ── 3. Histogram of Y ────────────────────────────────────────────────────────
ax = axes[2]
ax.hist(df[Y_COL], bins=10, color=PALETTE['test'],
        edgecolor='white', alpha=0.85)
ax.set_title(f'Distribution of {Y_COL}', fontweight='bold')
ax.set_xlabel(Y_COL)
ax.set_ylabel('Frequency')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

fig.suptitle('Exploratory Data Visualisation', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(f'{OUT_DIR}01_eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figure saved.')

## 3. 📈 Model Building (Professional)

Steps:
- **80 / 20** train-test split (stratification not applicable for continuous targets).
- Fit `LinearRegression` from scikit-learn (Ordinary Least Squares).
- Extract and display **slope (m)** and **intercept (b)** so the model is fully interpretable.

In [ ]:
# ── Feature / target arrays ───────────────────────────────────────────────────
X = df[[X_COL]].values  # 2-D array required by sklearn
y = df[Y_COL].values

# ── Train-test split (80 / 20, reproducible) ──────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Training samples : {len(X_train)}')
print(f'Test     samples : {len(X_test)}')

# ── Fit model ─────────────────────────────────────────────────────────────────
model = LinearRegression()
model.fit(X_train, y_train)

m = model.coef_[0]       # slope
b = model.intercept_     # intercept

print(f'\n📐 Fitted model : Y = {m:,.2f} · X + {b:,.2f}')
print(f'   Slope  (m)  : {m:,.4f}')
print(f'   Intercept(b): {b:,.4f}')

## 4. 📉 Predictions & Evaluation

In [ ]:
# ── Predictions ───────────────────────────────────────────────────────────────
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

# ── Metrics ───────────────────────────────────────────────────────────────────
metrics = {
    'Set'  : ['Train', 'Test'],
    'R²'   : [r2_score(y_train, y_pred_train),
               r2_score(y_test,  y_pred_test)],
    'MSE'  : [mean_squared_error(y_train, y_pred_train),
               mean_squared_error(y_test,  y_pred_test)],
    'RMSE' : [np.sqrt(mean_squared_error(y_train, y_pred_train)),
               np.sqrt(mean_squared_error(y_test,  y_pred_test))]
}

metrics_df = pd.DataFrame(metrics)
metrics_df.set_index('Set', inplace=True)

metrics_df.style \
    .format({'R²': '{:.4f}', 'MSE': '{:,.0f}', 'RMSE': '{:,.2f}'}) \
    .background_gradient(cmap='Greens', subset=['R²']) \
    .background_gradient(cmap='Reds_r', subset=['RMSE'])

## 5. 📊 Regression Line Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# ── Scatter: train points ─────────────────────────────────────────────────────
ax.scatter(X_train, y_train, color=PALETTE['train'],
           edgecolors='white', s=80, linewidths=0.7,
           alpha=0.9, label='Train set', zorder=3)

# ── Scatter: test points ──────────────────────────────────────────────────────
ax.scatter(X_test, y_test, color=PALETTE['test'],
           edgecolors='white', s=80, linewidths=0.7,
           alpha=0.9, label='Test set', zorder=3)

# ── Best-fit regression line ──────────────────────────────────────────────────
x_line = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
y_line = model.predict(x_line)
ax.plot(x_line, y_line, color=PALETTE['line'],
        linewidth=2.5, label=f'Regression line\ny = {m:,.0f}x + {b:,.0f}')

ax.set_title(f'Simple Linear Regression – {X_COL} vs {Y_COL}',
             fontweight='bold', pad=12)
ax.set_xlabel(X_COL)
ax.set_ylabel(Y_COL)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.4)

# Annotate R²
r2_test = r2_score(y_test, y_pred_test)
ax.text(0.04, 0.93, f'Test R² = {r2_test:.4f}',
        transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8),
        fontsize=11)

plt.tight_layout()
fig.savefig(f'{OUT_DIR}02_regression_line.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figure saved.')

## 6. 🔥 Standout Diagnostic Plots

Three plots that separate a professional notebook from a basic one:

| Plot | What to look for |
|---|---|
| **Residuals vs Fitted** | Points randomly scattered around 0 → assumptions hold |
| **Predicted vs Actual** | Points close to the 45° diagonal → accurate predictions |
| **Correlation matrix** | Quantifies the linear relationship |


In [ ]:
residuals_train = y_train - y_pred_train
residuals_test  = y_test  - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: Residuals vs Fitted ───────────────────────────────────────────────
ax = axes[0]
ax.scatter(y_pred_train, residuals_train, color=PALETTE['train'],
           edgecolors='white', s=70, alpha=0.85, label='Train')
ax.scatter(y_pred_test, residuals_test, color=PALETTE['test'],
           edgecolors='white', s=70, alpha=0.85, label='Test')
ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_title('Residuals vs Fitted', fontweight='bold')
ax.set_xlabel('Fitted Values')
ax.set_ylabel('Residuals')
ax.legend()

# ── Plot 2: Predicted vs Actual ───────────────────────────────────────────────
ax = axes[1]
all_actual    = np.concatenate([y_train, y_test])
all_predicted = np.concatenate([y_pred_train, y_pred_test])
colors_pts    = ([PALETTE['train']] * len(y_train) +
                 [PALETTE['test']]  * len(y_test))

ax.scatter(all_actual, all_predicted, c=colors_pts,
           edgecolors='white', s=70, alpha=0.85)
# 45° perfect-prediction line
lim_min = min(all_actual.min(), all_predicted.min()) * 0.97
lim_max = max(all_actual.max(), all_predicted.max()) * 1.03
ax.plot([lim_min, lim_max], [lim_min, lim_max],
        color='red', linestyle='--', linewidth=1.5, label='Perfect prediction')
ax.set_title('Predicted vs Actual', fontweight='bold')
ax.set_xlabel('Actual Values')
ax.set_ylabel('Predicted Values')
ax.legend()

# ── Plot 3: Residual distribution ─────────────────────────────────────────────
ax = axes[2]
ax.hist(residuals_train, bins=8, color=PALETTE['train'],
        edgecolor='white', alpha=0.75, label='Train')
ax.hist(residuals_test, bins=5, color=PALETTE['test'],
        edgecolor='white', alpha=0.75, label='Test')
ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_title('Residual Distribution', fontweight='bold')
ax.set_xlabel('Residual')
ax.set_ylabel('Frequency')
ax.legend()

fig.suptitle('Diagnostic Plots', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(f'{OUT_DIR}03_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figure saved.')

In [ ]:
# ── Linearity check: Pearson correlation ──────────────────────────────────────
corr = df[X_COL].corr(df[Y_COL])
print(f'Pearson Correlation Coefficient (r) = {corr:.4f}')
print(f'r² = {corr**2:.4f}  (explains {corr**2*100:.1f}% of variance)')

fig, ax = plt.subplots(figsize=(7, 5))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.4f', cmap='coolwarm',
            linewidths=0.5, ax=ax, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix', fontweight='bold')
plt.tight_layout()
fig.savefig(f'{OUT_DIR}04_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 🧠 Model Interpretation

### 7.1 Equation Interpretation

In [ ]:
r2_train = r2_score(y_train, y_pred_train)
r2_test  = r2_score(y_test,  y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print('=' * 60)
print('  MODEL INTERPRETATION REPORT')
print('=' * 60)
print(f'\n📐 Equation: {Y_COL} = {m:,.2f} × {X_COL} + {b:,.2f}')
print()
print(f'🔑 Slope  (m) = ${m:,.2f}')
print(f'   → For every additional year of experience,')
print(f'     salary increases by approximately ${m:,.2f}.')
print()
print(f'🔑 Intercept (b) = ${b:,.2f}')
print(f'   → A theoretical employee with 0 years of experience')
print(f'     is predicted to earn ${b:,.2f} (use with caution')
print(f'     outside the training range).')
print()
print(f'📊 R² (Train) = {r2_train:.4f} → model explains {r2_train*100:.1f}% of')
print(f'              variance in training salaries.')
print(f'📊 R² (Test)  = {r2_test:.4f} → model explains {r2_test*100:.1f}% of')
print(f'              variance in unseen test salaries.')
print()
print(f'📏 RMSE (Test) = ${rmse_test:,.2f}')
print(f'   → Average prediction error on test set.')
print()
print(f'✅ Business Insight:')
print(f'   The model is highly reliable for employees with')
print(f'   {df[X_COL].min():.1f}–{df[X_COL].max():.1f} years of experience.')
print(f'   Predictions beyond this range may be unreliable')
print(f'   (extrapolation risk).')
print('=' * 60)

### 7.2 Limitations & When NOT to Use This Model

| Limitation | Detail |
|---|---|
| **Extrapolation** | Predictions outside `YearsExperience` = [1.1, 10.5] are unreliable |
| **Single feature** | Salary is influenced by many factors (role, location, company size) not captured here |
| **Non-linearity** | If the true relationship curves (e.g., diminishing returns at senior levels), a polynomial or tree model would be better |
| **Small dataset** | Only 30 samples – model variance is higher than with a larger dataset |
| **Outliers** | OLS is sensitive to salary outliers; robust regression may be preferred in noisy real-world data |
| **Homoscedasticity** | If residual spread increases with salary, weighted regression is more appropriate |

## 8. 📈 Bonus: Feature Scaling Experiment

For simple linear regression with a single feature, `StandardScaler` does **not** change R² or MSE – the model is scale-invariant.
This experiment confirms that mathematically and is useful to demonstrate when comparing models or when preparing for regularised regression (Ridge, Lasso).

In [ ]:
# ── Scale X ───────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ── Train scaled model ────────────────────────────────────────────────────────
model_scaled = LinearRegression()
model_scaled.fit(X_train_scaled, y_train)

y_pred_test_scaled = model_scaled.predict(X_test_scaled)

r2_scaled   = r2_score(y_test, y_pred_test_scaled)
mse_scaled  = mean_squared_error(y_test, y_pred_test_scaled)
rmse_scaled = np.sqrt(mse_scaled)

# ── Comparison table ──────────────────────────────────────────────────────────
comparison_df = pd.DataFrame({
    'Model'      : ['Original (unscaled)', 'Scaled (StandardScaler)'],
    'R²'         : [r2_test, r2_scaled],
    'MSE'        : [mean_squared_error(y_test, y_pred_test), mse_scaled],
    'RMSE'       : [rmse_test, rmse_scaled],
    'Slope (m)'  : [model.coef_[0],
                    model_scaled.coef_[0]],
    'Intercept'  : [model.intercept_, model_scaled.intercept_]
})
comparison_df.set_index('Model', inplace=True)

print('Scaling Experiment – Performance Comparison')
comparison_df.style \
    .format({'R²': '{:.4f}', 'MSE': '{:,.0f}',
             'RMSE': '{:,.2f}', 'Slope (m)': '{:,.4f}',
             'Intercept': '{:,.2f}'}) \
    .background_gradient(cmap='Greens', subset=['R²'])

In [ ]:
# ── Side-by-side regression lines ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (title, X_tr, X_te, mdl, fmt) in zip(
    axes,
    [
        ('Original (Unscaled)', X_train, X_test, model,
         lambda x: x),
        ('StandardScaler Applied', X_train_scaled, X_test_scaled,
         model_scaled, lambda x: scaler.transform([[x]])[0][0])
    ]
):
    ax.scatter(X_tr, y_train, color=PALETTE['train'],
               edgecolors='white', s=70, alpha=0.85, label='Train')
    ax.scatter(X_te, y_test, color=PALETTE['test'],
               edgecolors='white', s=70, alpha=0.85, label='Test')

    x_rng = np.linspace(X_tr.min(), X_tr.max(), 200).reshape(-1, 1)
    ax.plot(x_rng, mdl.predict(x_rng),
            color=PALETTE['line'], linewidth=2.5)

    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(Y_COL)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.legend()

fig.suptitle('Regression Lines: Original vs Scaled Feature',
             fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{OUT_DIR}05_scaling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figure saved.')
print('\n📝 Note: R² is identical in both models – confirming scale-invariance for OLS.')

## ✅ Final Summary & Export

In [ ]:
# ── Print final metrics summary ───────────────────────────────────────────────
print('=' * 55)
print('  FINAL MODEL SUMMARY')
print('=' * 55)
print(f'  Dataset          : Salary_Data.csv ({len(df)} samples)')
print(f'  Feature (X)      : {X_COL}')
print(f'  Target  (Y)      : {Y_COL}')
print(f'  Train / Test     : {len(X_train)} / {len(X_test)}')
print('-' * 55)
print(f'  Equation         : Salary = {m:,.2f} × Exp + {b:,.2f}')
print(f'  R² (Train)       : {r2_train:.4f}')
print(f'  R² (Test)        : {r2_test:.4f}')
print(f'  RMSE (Test)  ($) : {rmse_test:,.2f}')
print(f'  Pearson r        : {corr:.4f}')
print('=' * 55)

# ── Save predictions CSV ──────────────────────────────────────────────────────
pred_df = pd.DataFrame({
    X_COL             : X_test.flatten(),
    'Actual_Salary'   : y_test,
    'Predicted_Salary': y_pred_test.round(2),
    'Residual'        : (y_test - y_pred_test).round(2)
})
pred_csv_path = f'{OUT_DIR}predictions.csv'
pred_df.to_csv(pred_csv_path, index=False)
print(f'\n📥 Predictions saved → {pred_csv_path}')
pred_df

In [ ]:
# ── Styled final metrics table ────────────────────────────────────────────────
final_metrics = pd.DataFrame({
    'Metric'  : ['R² (Train)', 'R² (Test)', 'MSE (Test)',
                 'RMSE (Test)', 'Slope (m)', 'Intercept (b)',
                 'Pearson r'],
    'Value'   : [f'{r2_train:.4f}', f'{r2_test:.4f}',
                 f'{mean_squared_error(y_test, y_pred_test):,.0f}',
                 f'${rmse_test:,.2f}',
                 f'${m:,.2f} / year',
                 f'${b:,.2f}',
                 f'{corr:.4f}'],
    'Interpretation': [
        'Variance explained on training data',
        'Variance explained on unseen data',
        'Average squared error',
        'Average prediction error',
        'Salary increase per year of experience',
        'Predicted salary at 0 years (extrapolation)',
        'Strength of linear relationship'
    ]
})

final_metrics.set_index('Metric').style \
    .set_caption('📊 Linear Regression – Key Metrics at a Glance') \
    .set_properties(**{'text-align': 'left'})

---

## 🎓 Conclusion

This notebook demonstrated a **complete, production-ready Simple Linear Regression pipeline**:

1. **Strong linear relationship** confirmed visually and via Pearson r ≈ 0.98.
2. **Model equation** quantifies exactly how salary scales with experience.
3. **High R²** on both train and test indicates the model generalises well with no signs of overfitting.
4. **Diagnostic plots** confirm residuals are approximately normally distributed and randomly scattered — OLS assumptions are satisfied.
5. **Scaling experiment** confirms OLS is scale-invariant (useful to know before moving to regularised models).

> **Next steps to improve the model:**  
> Add more features (job title, location, industry) → move to Multiple Linear Regression or Gradient Boosting for non-linear relationships.

---
*Notebook generated for: `Salary_Data.csv` | Framework: scikit-learn | Author: Data Science Pipeline*